In [ ]:
from __future__ import division
import numpy as np
from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt
import time
import pickle

In [ ]:
class Populations(object):

    def __init__(self,nReps,nInds,nLoci,ploidy=45,mu=0.0001,beneficial_selcoef=0.1, deleterious_selcoef = 0.1, \
                 beneficial_prop = 0.5, model_version ='ADD'):

        self.nReps = nReps
        self.nInds = nInds
        self.nLoci = nLoci
        
        self.soma = np.zeros((nReps,nInds,nLoci),dtype='int')
        self.soma_beneficial = np.zeros((nReps,nInds,nLoci),dtype='int') # store the beneficial mutations
        self.soma_deleterious = np.zeros((nReps,nInds,nLoci),dtype='int') # store the deleterious mutations
        
        self.germ = np.zeros((nReps,nInds,nLoci),dtype='int')
        self.germ_beneficial = np.zeros((nReps,nInds,nLoci),dtype='int')
        self.germ_deleterious = np.zeros((nReps,nInds,nLoci),dtype='int')        
        
        self.ploidy = ploidy
        self.mu = mu
        self.beneficial_selcoef = beneficial_selcoef
        self.deleterious_selcoef = deleterious_selcoef
        
        self.beneficial_prop = beneficial_prop
        
        self.current_step = 0
        self.model_version = model_version
        

    def fitness(self):
        """return a numpy array containing the fitness for each individual in each population"""
        
        # need to add beneficial mutations
        bene_ws = (self.soma_beneficial.astype('float')/self.ploidy)*self.beneficial_selcoef
        dele_ws = (self.soma_deleterious.astype('float')/self.ploidy)*self.deleterious_selcoef

        if self. model_version == 'ADD':
            fitnesses = 1 + np.sum(bene_ws,axis=2) -np.sum(dele_ws, axis =2)
            
        elif self. model_version == 'MUL':
            each_locus = 1 + bene_ws - dele_ws
            fitnesses = np.prod(each_locus, axis =2)
            
        fitnesses[fitnesses <= 0] = 0  
        return fitnesses
    
    
    def relative_fitness(self):
        """return a numpy array containing each individual's relative fitness"""
        w = self.fitness()
        total_w = np.sum(w,axis=1)
        totalw = np.expand_dims(total_w,axis=1)
        relfit = w/totalw
        return relfit    
    
    
    def germ_fitness(self):
        """return a numpy array containing the fitness for each individual in each population"""
        
        # need to add beneficial mutations
        bene_gs = (self.germ_beneficial.astype('float')/2)*self.beneficial_selcoef
        dele_gs = (self.germ_deleterious.astype('float')/2)*self.deleterious_selcoef

        if self. model_version == 'ADD':
            germ_fit = 1 + np.sum(bene_gs,axis=2) -np.sum(dele_gs, axis =2)
            
        elif self. model_version == 'MUL':
            each_germ_locus = 1 + bene_gs - dele_gs
            germ_fit = np.prod(each_germ_locus, axis =2)
            
        germ_fit[germ_fit <= 0] = 0  
        return germ_fit
    
    
    
    #Wright_Fisher model
    def mutate_all_before(self):
        """Mutate all individuals in the population with mutation rate mu.
        Wright-Fisher model method. Use this one instead of mutate_all if you want to mutate the population before selection."""
        
        wt_soma = self.ploidy - self.soma   # the still mutable alleles in each loci
        wt_germ = 2 - self.germ
        
        soma_mutations = np.random.binomial(wt_soma,self.mu) # generate new mutations
        germ_mutations = np.random.binomial(wt_germ,self.mu) 
        
        soma_bene_mu = np.random.binomial(soma_mutations, self.beneficial_prop)  # How many of the new mutations are beneficial
        soma_dele_mu = soma_mutations - soma_bene_mu  # and deleterious
        
        germ_bene_mu = np.random.binomial(germ_mutations, self.beneficial_prop)
        germ_dele_mu = germ_mutations - germ_bene_mu
        
        self.soma += soma_mutations   # update the soma and germ (all mutations, beneficial and deleterious mutations)
        self.germ += germ_mutations  # how many mutations have been generated
        
        self.soma_beneficial += soma_bene_mu   
        self.germ_beneficial += germ_bene_mu      
        
        self.soma_deleterious += soma_dele_mu   
        self.germ_deleterious += germ_dele_mu  
        return    
    
    
    def pick_parents_all(self):
        """Randomly choose N parents to produce offspring to populate the next generation.
        Each individual's probability of being chosen is weighted by its relative fitness.
        Wright-Fisher model method."""
        nReps,nInds = self.soma.shape[0:2]        
        relfit = self.relative_fitness()

        csrelfit = np.cumsum(relfit,axis=1)
        randvals = np.random.random((nReps,nInds))
        parents = map(np.searchsorted,csrelfit,randvals)
        return parents    
    
    
    def mitosis_all(self,parents):
        """Generate mitotic offspring from all individuals selected as parents.
        Wright-Fisher model method."""
        nReps,nInds = self.soma.shape[0:2]        
        rReps = np.ones((nReps,nInds),dtype='int')*np.expand_dims(np.arange(nReps),axis=1)
        self.soma = self.soma[rReps,parents,]
        self.germ = self.germ[rReps,parents,]
        
        self.soma_beneficial = self.soma_beneficial[rReps,parents,]
        self.germ_beneficial = self.germ_beneficial[rReps,parents,]        

        self.soma_deleterious = self.soma_deleterious[rReps,parents,]
        self.germ_deleterious = self.germ_deleterious[rReps,parents,] 
        
        return    
    
    
    def amitosis_all(self,parents):
        """Generate amitotic offspring from all individuals selected as parents. Only one
        amitotic offspring is generated from each parent, so this method does not reflect
        the reciprocity of amitosis.
        Wright-Fisher model method."""
        
        # may consider multi hypergeometric distribution
        
        nReps,nInds = self.soma.shape[0:2]        
        rReps = np.ones((nReps,nInds),dtype='int')*np.expand_dims(np.arange(nReps),axis=1)
        
        good = (self.ploidy-self.soma[rReps,parents,])*2  # wild type alleles in each locus
        bad = self.soma[rReps, parents, ]*2  # mutated alleles (bene+dele) in each locus
        
        bad_bene = self.soma_beneficial[rReps,parents,]*2
        bad_dele = self.soma_deleterious[rReps,parents,]*2
        
        self.soma = self.ploidy - np.random.hypergeometric(good, bad, self.ploidy) # np.random.hypergeometric(good, bad, 
                                                             # \self.ploidy) is the number of wild type alleles in each locus.
            # self.ploidy - np.random.hypergeometric(good, bad, self.ploidy) is the number of mutated alleles (bene + dele) in
            # each locus
        
        self.soma_beneficial[self.soma!=0]= np.random.hypergeometric(bad_bene[self.soma!=0], bad_dele[self.soma!=0], \
                                                                                 self.soma[self.soma!=0])
        self.soma_beneficial[self.soma==0]=0
        
        self.soma_deleterious = self.soma - self.soma_beneficial


        self.germ = self.germ[rReps,parents,]
        self.germ_beneficial = self.germ_beneficial[rReps, parents,]
        self.germ_deleterious = self.germ_deleterious[rReps, parents,]

        return    
    
    
    def wright_fisher_step(self,asex_type='amitosis'):
        """Take populations through a single Wright-Fisher model generation"""
        self.mutate_all_before()
        parents = self.pick_parents_all()
        if asex_type == 'amitosis':
            self.amitosis_all(parents)
        elif asex_type == 'mitosis':
            self.mitosis_all(parents)
        self.current_step += 1
        return    
    
    
    # Asexul reproduction_WF model and Moran model
    
    def step(self,model='M',asex_type='amitosis',sex_freq=None):
        """Take populations through one time step if model='M', or one generation if model='WF'"""
        if model == 'M':
            self.moran_step(asex_type)
        elif model == 'WF':
            self.wright_fisher_step(asex_type)
        return 

    
    
    def make_gametes(self, parents):

        rReps = np.ones((self.nReps,self.nInds),dtype='int')*np.expand_dims(np.arange(self.nReps),axis=1)
        gametes = 1- np.random.hypergeometric(2-self.germ[rReps,parents,],self.germ[rReps,parents,],1) 
        # np.random.hypergeometric(2-self.germ[rReps,parents,],self.germ[rReps,parents,],1) stores how many wild type alleles 
        # remains in each locus. 1- np.random.hypergeometric(2-self.germ[rReps,parents,],self.germ[rReps,parents,],1) stores 
        # the number of mutations in each locus.
        
        gametes_bene = np.zeros((self.nReps,self.nInds,self.nLoci),dtype='int')  # initialize beneficial and deleterious 
        gametes_dele = np.zeros((self.nReps,self.nInds,self.nLoci),dtype='int')  # in gametes
        
        gametes_bene[gametes!=0] = np.random.hypergeometric(self.germ_beneficial[rReps, parents,][gametes!=0], \
                                                            self.germ_deleterious[rReps, parents,][gametes!=0],\
                                                            gametes[gametes!=0])
        # For locus where gametes ==0, it means that locus has no mutations. Thus also have no beneficial and deleterious 
        # mutations.
        
        gametes_dele = gametes - gametes_bene
        
        return gametes, gametes_bene, gametes_dele
    

    def make_zygotes(self, random_mating=False):

        if random_mating == False:
            parents = self.pick_parents_all()

            these = self.make_gametes(parents)
            those = self.make_gametes(parents) 

        else: 
            first_parents = self.pick_parents_all()
            second_parents = self.pick_parents_all()

            these = self.make_gametes(first_parents)
            those = self.make_gametes(second_parents)           

        these_gametes = these[0]
        these_gametes_bene = these[1]
        these_gametes_dele = these[2]
            
        those_gametes = those[0]
        those_gametes_bene = those[1]
        those_gametes_dele = those[2]

        zygotes = these_gametes + those_gametes
        zy_bene = these_gametes_bene + those_gametes_bene
        zy_dele = these_gametes_dele + those_gametes_dele
        
        return zygotes, zy_bene, zy_dele
    
    
    
    def sex(self,random_mating=False):
        
        self.mutate_all_before()

        zy = self.make_zygotes(random_mating)  
        
        zygotes = zy[0]
        zy_bene = zy[1]
        zy_dele = zy[2]
        
        self.germ = zygotes  # the germline genome will be the same as the zygotes.
        self.germ_beneficial = zy_bene
        self.germ_deleterious = zy_dele

        self.soma = np.zeros((self.nReps,self.nInds,self.nLoci),dtype='int')   # initialize the soma before amplication
        self.soma_beneficial = np.zeros((self.nReps,self.nInds,self.nLoci),dtype='int')
        self.soma_deleterious = np.zeros((self.nReps,self.nInds,self.nLoci),dtype='int')
        
        self.soma[self.germ ==0] =0  # if for some loci self.germ ==0, i.e., zygotes ==0 in these loci, then it means there
        self.soma_beneficial[self.germ ==0] =0   # are no mutations in these loci. soma will also be 0 (all alleles are WT)
        self.soma_deleterious[self.germ ==0] =0
        
         # if self.germ ==1 or 2: check it is beneficial or deleterious
       
        for i in range(self.nReps):
            for j in range(self.nInds):
                for k in range(self.nLoci):
                    if self.germ[i][j][k] ==1 and self.germ_beneficial[i][j][k]==1:  # het: beneficial
#                         bene_het_pos.append([i,j,k])
                        self.soma[i][j][k] = np.random.binomial(self.ploidy, 0.5) #how many new beneficial mutations will generate
                        self.soma_beneficial[i][j][k] = self.soma[i][j][k]
                        self.soma_deleterious[i][j][k] =0
    
                    elif self.germ[i][j][k]==1 and self.germ_deleterious[i][j][k]==1: # het: deleterious
#                         dele_het_pos.append([i,j,k])
                        self.soma[i][j][k] = np.random.binomial(self.ploidy, 0.5)
                        self.soma_deleterious[i][j][k] = self.soma[i][j][k]
                        self.soma_beneficial[i][j][k] =0

                    elif self.germ[i][j][k]==2: # this locus is mutation fixed.
                        self.soma[i][j][k]=self.ploidy  # mutation fixed in this locus
                        if self.germ_beneficial[i][j][k]==2: # this locus is homozygous for beneficial mutations
                            self.soma_beneficial[i][j][k]=self.ploidy
                        elif self.germ_deleterious[i][j][k]==2: # this locus is homozygous for deleterious mutations
                            self.soma_deleterious[i][j][k]=self.ploidy
                        else: # one is beneficial and another is deleterious
                            if self.germ_beneficial[i][j][k]!=1 or self.germ_deleterious[i][j][k]!=1:
                                print 'ERROR'
                            self.soma_beneficial[i][j][k] =np.random.binomial(self.ploidy, 0.5) # how many new beneficial mutations will generate
                            self.soma_deleterious[i][j][k] = self.ploidy - self.soma_beneficial[i][j][k]  
               
        self.current_step +=1
        return     
    
    
    def get_results(self):
        """calculate stuff like mean fitness, Gst, and number of fixed mutations"""
        W = self.fitness()
        log_W = np.log(W)

        mW = np.nanmean(W)
        log_mW = np.log(mW)
        sdW = np.nanstd(W)
        semW = stats.sem(W,None)

        mlog_W = np.nanmean(log_W)
        sdlog_W = np.nanstd(log_W)
        semlog_W = stats.sem(log_W,None)

        total_log_var = []
        total_fit_var = []
        for i in range(self.nReps):
            log_var = np.var(log_W[i])
            fit_var = np.var(W[i])

            total_log_var.append(log_var)
            total_fit_var.append(fit_var)

        varlog_W = np.nanmean(total_log_var)
        varW = np.nanmean(total_fit_var)


        GF = self.germ_fitness()
        log_GF = np.log(GF)

        mGF = np.nanmean(GF)
        log_mGF = np.log(mGF)
        sdGF = np.nanstd(GF)
        semGF = stats.sem(GF,None)

        mlog_GF = np.nanmean(log_GF)
        sdlog_GF = np.nanstd(log_GF)
        semlog_GF = stats.sem(log_W,None)

        total_log_var_g = []
        total_fit_var_g = []
        for i in range(self.nReps):
            log_var_g = np.var(log_GF[i])
            fit_var_g = np.var(GF[i])

            total_log_var_g.append(log_var_g)
            total_fit_var_g.append(fit_var_g)

        varlog_GF = np.nanmean(total_log_var_g)
        varGF = np.nanmean(total_fit_var_g)        

        return [mW,log_mW, sdW,semW,mlog_W, sdlog_W, semlog_W, varlog_W, varW, \
                mGF,log_mGF, sdGF,semGF,mlog_GF, sdlog_GF, semlog_GF, varlog_GF, varGF]
    
    


    @staticmethod
    def monitor_population_mutation(soma_array, ploidy):
        '''Monitor the mutation dynamics in population level'''

        nReps, nInds, nLoci = soma_array.shape[0:3]

        fixed = np.sum(np.sum(soma_array==ploidy,axis=1)==nInds,axis=1)
        all_wt_site = np.sum(np.sum(soma_array==0,axis=1)==nInds,axis=1)

        poly_site = nLoci-fixed-all_wt_site


        overall_mu_freq = np.sum(soma_array, axis =1)/(ploidy*nInds)

        value_all_poly_freq_pl = []
        value_all_poly_var_pl = []


        for i in range(nReps):
            pop_poly_freq_pl = []
            # pop_poly_var_pl = []

            for j in range(nLoci):
                if overall_mu_freq[i][j]!= 0 and overall_mu_freq[i][j]!= 1:
                    pop_poly_freq_pl.append(overall_mu_freq[i][j])

            if len(pop_poly_freq_pl) == 0:
                value_pop_poly_freq = 0
                value_pop_poly_var = 0
            elif len(pop_poly_freq_pl) == 1:
                value_pop_poly_freq = np.nanmean(pop_poly_freq_pl)
                value_pop_poly_var = 0
            else:
                value_pop_poly_freq = np.nanmean(pop_poly_freq_pl)
                value_pop_poly_var = np.var(pop_poly_freq_pl)


            value_all_poly_freq_pl.append(value_pop_poly_freq)
            value_all_poly_var_pl.append(value_pop_poly_var)


        meanPopFM = np.nanmean(fixed)
        stdPopFM = np.nanstd(fixed)
        sePopFM = stdPopFM/(len(fixed)**0.5)
        semPopFM = stats.sem(fixed,None)

        meanPopPM = np.nanmean(poly_site)
        stdPopPM = np.nanstd(poly_site)
        sePopPM = stdPopPM/(len(poly_site)**0.5)
        semPopPM = stats.sem(poly_site,None)     

        meanPopPF = np.nanmean(value_all_poly_freq_pl)   
        stdPopPF = np.nanstd(value_all_poly_freq_pl)
        sePopPF = stdPopPF/(len(value_all_poly_freq_pl)**0.5)
        semPopPF =  stats.sem(value_all_poly_freq_pl,None)

        meanPopPV = np.nanmean(value_all_poly_var_pl)
        stdPopPV = np.nanstd(value_all_poly_var_pl)
        sePopPV = stdPopPV/(len(value_all_poly_var_pl)**0.5)
        semPopPV = stats.sem(value_all_poly_var_pl,None)

        return [meanPopFM, stdPopFM, sePopFM, semPopFM, meanPopPM, stdPopPM, sePopPM, semPopPM, \
        meanPopPF, stdPopPF, sePopPF, semPopPF, meanPopPV, stdPopPV, sePopPV, semPopPV]


    @staticmethod
    def monitor_individual_mutation(soma_array, ploidy):
        '''Monitor the mutation dynamics in each site in each individual in each population'''
        # nReps, nInds, nLoci = self.soma.shape[0:3]
        # ploidy = self.ploidy
        
        all_fix_site = []
        all_poly_site = []
        
        all_poly_freq = []
        all_poly_var = []
        
        nReps, nInds, nLoci = soma_array.shape[0:3]

        for i in range(nReps):
            pop_fix_site = []
            pop_poly_site = []
            
            pop_poly_freq = []
            pop_poly_var = []
            
            for j in range(nInds):
                ind_fix_site = 0
                ind_poly_site = 0
            
                ind_poly_freq = []
#                 ind_poly_var = 0
                
                for k in range(nLoci): # loci level
                    if soma_array[i][j][k] == ploidy:
                        ind_fix_site +=1
                    elif soma_array[i][j][k]>0 and soma_array[i][j][k]<ploidy:
                        ind_poly_site +=1
                        ind_poly_freq.append(soma_array[i][j][k]/ploidy)  # the freq in each poly site
                        
                ind_poly_freq_mean = np.nanmean(ind_poly_freq) 
                # print '1', ind_poly_freq_mean
                        
                if len(ind_poly_freq) ==0 or len(ind_poly_freq) ==1: 
                    ind_poly_var = 0
                else:
                    ind_poly_var = np.var(ind_poly_freq) # this is the variance of each individual
                    
                
                pop_fix_site.append(ind_fix_site) # no. of fixed mutation site in each individual in a population
                pop_poly_site.append(ind_poly_site) # no. of poly mutation site in each individual in a population
                
                pop_poly_freq.append(ind_poly_freq_mean) # the poly freq of each individual in a population
                pop_poly_var.append(ind_poly_var) # the variance of each individual in a population
                
            
            all_fix_site.append(np.nanmean(pop_fix_site)) # mean no. of fixed mutation site in a population
            all_poly_site.append(np.nanmean(pop_poly_site)) # mean no. of ploy mutation site in a population
            all_poly_freq.append(np.nanmean(pop_poly_freq)) # mean poly freq in a population
            all_poly_var.append(np.nanmean(pop_poly_var)) # mean poly variance in a population
            
            
        mean_fix_site = np.nanmean(all_fix_site)
        std_fix_site = np.nanstd(all_fix_site)
        se_fix_site = std_fix_site/(len(all_fix_site)**0.5)
        sem_fix_site = stats.sem(all_fix_site,None)

        mean_poly_site = np.nanmean(all_poly_site)
        std_poly_site = np.nanstd(all_poly_site)
        se_poly_site = std_poly_site/(len(all_poly_site)**0.5)
        sem_poly_site = stats.sem(all_poly_site,None)

        mean_poly_freq = np.nanmean(all_poly_freq)
        std_poly_freq = np.nanstd(all_poly_freq)
        se_poly_freq = std_poly_freq/(len(all_poly_freq)**0.5)
        sem_poly_freq = stats.sem(all_poly_freq,None)

        mean_poly_var = np.nanmean(all_poly_var)
        std_poly_var = np.nanstd(all_poly_var)
        se_poly_var = std_poly_var/(len(all_poly_var)**0.5)
        sem_poly_var = stats.sem(all_poly_var,None)
        

        return [mean_fix_site, std_fix_site, se_fix_site, sem_fix_site, mean_poly_site, std_poly_site, se_poly_site, sem_poly_site, \
        mean_poly_freq, std_poly_freq, se_poly_freq, sem_poly_freq, mean_poly_var, std_poly_var, se_poly_var, sem_poly_var]    
    



    def simulate(self,stepcount,model='M',asex_type='amitosis',sex_freq=None,random_mating=False,strides=10):
        results = [self.get_results()]

        overall_ind_mutations = [self.monitor_individual_mutation(self.soma, self.ploidy)]
        overall_pop_mutations = [self.monitor_population_mutation(self.soma, self.ploidy)]

        beneficial_ind_mutations = [self.monitor_individual_mutation(self.soma_beneficial, self.ploidy)]
        beneficial_pop_mutations = [self.monitor_population_mutation(self.soma_beneficial, self.ploidy)]

        deleterious_ind_mutations = [self.monitor_individual_mutation(self.soma_deleterious, self.ploidy)]
        deleterious_pop_mutations = [self.monitor_population_mutation(self.soma_deleterious, self.ploidy)]
        
        start = time.time()
   
        while self.current_step <= stepcount:
            if (sex_freq != None) and (self.current_step%sex_freq == 0):
                self.sex(random_mating)
                self.sex_gen.append(self.current_step)
              
            else:
                self.step(model,asex_type,sex_freq)
                
            if self.current_step%strides == 0:
#                 print 'STEP', self.current_step
                results.append(self.get_results())

                overall_ind_mutations.append(self.monitor_individual_mutation(self.soma, self.ploidy))
                overall_pop_mutations.append(self.monitor_population_mutation(self.soma, self.ploidy))

                beneficial_ind_mutations.append(self.monitor_individual_mutation(self.soma_beneficial, self.ploidy))
                beneficial_pop_mutations.append(self.monitor_population_mutation(self.soma_beneficial, self.ploidy))                

                deleterious_ind_mutations.append(self.monitor_individual_mutation(self.soma_deleterious, self.ploidy))
                deleterious_pop_mutations.append(self.monitor_population_mutation(self.soma_deleterious, self.ploidy))

                
        colnames = ['meanFit','log_meanFit','sdFit','semFit','mean_logFit','sdlog_Fit', 'semlog_Fit', 'varlog_Fit', 'varFit', \
                    'meanFit_Germ','log_meanFit_Germ','sdFit_Germ','semFit_Germ','mean_logFit_Germ','sdlog_Fit_Germ', 'semlog_Fit_Germ',\
                    'varlog_Fit_Germ', 'varFit_Germ']

        ind_mutation_colnames = ['meanFM', 'stdFM', 'seFM', 'semFM', 'meanPM', 'stdPM', 'sePM', 'semPM', \
                                 'meanPF','stdPF', 'sePF', 'semPF', 'meanPV', 'stdPV', 'sePV', 'semPV']
        
        pop_mutation_colnames = ['meanPopFM', 'stdPopFM', 'sePopFM', 'semPopFM', 'meanPopPM', 'stdPopPM', 'sePopPM', 'semPopPM',\
                                 'meanPopPF', 'stdPopPF', 'sePopPF', 'semPopPF', 'meanPopPV', 'stdPopPV', 'sePopPV', 'semPopPV']

        
        results = pd.DataFrame(np.array(results),columns=colnames)

        overall_ind_mutations = pd.DataFrame(np.array(overall_ind_mutations),columns=ind_mutation_colnames)
        overall_pop_mutations = pd.DataFrame(np.array(overall_pop_mutations),columns=pop_mutation_colnames) 

        beneficial_ind_mutations = pd.DataFrame(np.array(beneficial_ind_mutations),columns=ind_mutation_colnames)
        beneficial_pop_mutations = pd.DataFrame(np.array(beneficial_pop_mutations),columns=pop_mutation_colnames)

        deleterious_ind_mutations = pd.DataFrame(np.array(deleterious_ind_mutations),columns=ind_mutation_colnames)
        deleterious_pop_mutations = pd.DataFrame(np.array(deleterious_pop_mutations),columns=pop_mutation_colnames)

        
        end = time.time()
        print 'TOTAL TIME: ',end-start
        
        return results, overall_ind_mutations, overall_pop_mutations, \
        beneficial_ind_mutations, beneficial_pop_mutations, deleterious_ind_mutations, deleterious_pop_mutations
    
    
    def simulateNsave(self,outfile_1,outfile_2, outfile_3, outfile_4, outfile_5, outfile_6, outfile_7, \
        stepcount,model='M',asex_type='amitosis',sex_freq=None,random_mating=False,strides=10):
        """same as simulate method except results are written to the file specified by outfile"""
        results = self.simulate(stepcount,model,asex_type,sex_freq,random_mating,strides)
        
        result = results[0]
        result.to_csv(outfile_1,index=False)
        
        overall_ind_mutation = results[1]
        overall_ind_mutation.to_csv(outfile_2, index=False)

        overall_pop_mutation = results[2]
        overall_pop_mutation.to_csv(outfile_3, index=False)

        beneficial_ind_mutation = results[3]
        beneficial_ind_mutation.to_csv(outfile_4, index=False)

        beneficial_pop_mutation = results[4]
        beneficial_pop_mutation.to_csv(outfile_5, index=False)

        deleterious_ind_mutation = results[5]
        deleterious_ind_mutation.to_csv(outfile_6, index=False)

        deleterious_pop_mutation = results[6]
        deleterious_pop_mutation.to_csv(outfile_7, index=False)

        
        return